In [ ]:
# All package imports (run this cell first)
import sys
import json
from pathlib import Path

import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import PeftModel, get_peft_model, LoraConfig, TaskType

## Kernel check (use root .venv)

Run this cell first to confirm the notebook is using the project's root `.venv`.

In [ ]:
# Kernel verification
_venv_ok = "My-Crew-Manager" in sys.executable and ".venv" in sys.executable
print(f"Python: {sys.executable}")
print(f"Using root .venv: {'✓ Yes' if _venv_ok else '✗ No – select Kernel → Python (My-Crew-Manager .venv)'}")

# Model Part 2 - Backlog Generation

From structured Part 1 JSON to backlog text (Epic to Sub-Epic to User Story to Task).

This notebook expects an existing model2_part1_to_backlog.jsonl dataset in llms/fine_tune/dataset.

## Setup paths

In [ ]:
# Resolve AI root
_cwd = Path.cwd()
_ai_root = _cwd if (_cwd / "llms").exists() else (_cwd / "AI" if (_cwd / "AI").exists() else _cwd)
if str(_ai_root) not in sys.path:
    sys.path.insert(0, str(_ai_root))

FINE_TUNE_DIR = _ai_root / "llms" / "fine_tune"
DATASET_DIR = FINE_TUNE_DIR / "dataset"
TOKENIZED_DIR = FINE_TUNE_DIR / "tokenized"
OUTPUT_DIR = FINE_TUNE_DIR / "qwen_model2_backlog_lora"

print(f"AI root: {_ai_root}")
print(f"Dataset: {DATASET_DIR}")
print(f"Output: {OUTPUT_DIR}")

## Step 1: Import existing Model 2 dataset

Load the prepared `model2_part1_to_backlog.jsonl` dataset and preview sample records.

In [ ]:
# Import and preview Model 2 dataset
dataset_file = DATASET_DIR / "model2_part1_to_backlog.jsonl"
if not dataset_file.exists():
    raise FileNotFoundError(
        f"Missing dataset: {dataset_file}. Generate or convert datasets first."
    )

rows = []
for line in dataset_file.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line:
        continue
    rows.append(json.loads(line))

print(f"Loaded {len(rows)} examples from {dataset_file.name}")
print("Model 2 contract:")
print("- Prompt: Model 1 response text (Title, Summary, Roles, Features, Goals, Timeline)")
print("- Response: Backlog only in Goal -> User Story -> Task hierarchy")
print("- Task rule: exactly 2 tasks per goal story")
if rows:
    print("\nSample prompt snippet (Model 1 text):")
    print(rows[0]["prompt"][:280] + ("..." if len(rows[0]["prompt"]) > 280 else ""))
    print("\nSample backlog response snippet:")
    print(rows[0]["response"][:280] + ("..." if len(rows[0]["response"]) > 280 else ""))

## Step 2: Prepare tokenized dataset

In [ ]:
from llms.fine_tune.prepare_dataset import prepare_model2_dataset

tokenized_path = TOKENIZED_DIR / "tokenized_model2_qwen"
MAX_LENGTH = 768

if tokenized_path.exists():
    dataset = load_from_disk(str(tokenized_path))
    print(f"Loaded tokenized dataset from {tokenized_path}")
else:
    dataset = prepare_model2_dataset(
        model_name="qwen",
        max_length=MAX_LENGTH,
        output_dir=str(tokenized_path),
    )

split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Total dataset size: {len(dataset)}")
print(f"Train size: {len(train_dataset)}")
print(f"Eval size: {len(eval_dataset)}")

## Step 3: Load model & apply LoRA

In [ ]:
MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"
BATCH_SIZE = 2
EPOCHS = 5

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
if torch.cuda.is_available():
    model = model.to("cuda")

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Step 4: Train iteratively until target

Train in capped rounds, evaluate after each round, and stop early when quality targets are reached.

In [ ]:
import inspect
import math
import os
os.environ.setdefault("TENSORBOARD_LOGGING_DIR", str(OUTPUT_DIR / "logs"))

TARGET_EVAL_LOSS = 1.20
TARGET_PERPLEXITY = 3.50
MAX_ROUNDS = 8
EPOCHS_PER_ROUND = 1

training_kwargs = {
    "output_dir": str(OUTPUT_DIR),
    "per_device_train_batch_size": BATCH_SIZE,
    "num_train_epochs": EPOCHS_PER_ROUND,
    "logging_steps": 10,
    "save_steps": 50,
    "save_total_limit": 2,
    "eval_steps": 50,
    "fp16": torch.cuda.is_available(),
    "report_to": "none",
}
if "evaluation_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    training_kwargs["evaluation_strategy"] = "steps"
else:
    training_kwargs["eval_strategy"] = "steps"

training_args = TrainingArguments(**training_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

round_history = []
stop_reason = "max_rounds_reached"

for round_num in range(1, MAX_ROUNDS + 1):
    print(f"\n--- Round {round_num}/{MAX_ROUNDS} ---")
    train_result = trainer.train()

    pred_output = trainer.predict(eval_dataset, metric_key_prefix="eval")
    eval_metrics = pred_output.metrics
    eval_loss = eval_metrics.get("eval_loss") or eval_metrics.get("test_loss")
    perplexity = math.exp(eval_loss) if eval_loss is not None and eval_loss < 20 else float("inf")

    round_entry = {
        "round": round_num,
        "train_loss": float(train_result.training_loss),
        "eval_loss": float(eval_loss) if eval_loss is not None else None,
        "perplexity": float(perplexity),
    }
    round_history.append(round_entry)

    print(f"Train loss: {round_entry['train_loss']:.4f}")
    if eval_loss is not None:
        print(f"Eval loss: {eval_loss:.4f}")
    else:
        print("Eval loss: unavailable")
    print(f"Perplexity: {perplexity:.4f}" if math.isfinite(perplexity) else "Perplexity: inf")

    if eval_loss is not None and (eval_loss <= TARGET_EVAL_LOSS or perplexity <= TARGET_PERPLEXITY):
        stop_reason = "target_reached"
        print(
            f"Stopping early: eval_loss={eval_loss:.4f}, perplexity={perplexity:.4f}, "
            f"targets=({TARGET_EVAL_LOSS}, {TARGET_PERPLEXITY})"
        )
        break

valid_rounds = [r for r in round_history if r["eval_loss"] is not None and math.isfinite(r["eval_loss"])]
best_round = min(valid_rounds, key=lambda r: r["eval_loss"]) if valid_rounds else None

print("\n=== Iterative training summary ===")
print(f"Stop reason: {stop_reason}")
print(f"Rounds completed: {len(round_history)}")
if best_round:
    print(
        f"Best round: {best_round['round']} "
        f"(eval_loss={best_round['eval_loss']:.4f}, perplexity={best_round['perplexity']:.4f})"
    )
else:
    print("Best round: unavailable (no valid eval_loss)")
for r in round_history:
    print(
        f"Round {r['round']}: train_loss={r['train_loss']:.4f}, "
        f"eval_loss={r['eval_loss'] if r['eval_loss'] is not None else 'NA'}, "
        f"perplexity={r['perplexity']:.4f}"
    )

## Step 4.1: Evaluate model performance

Compute evaluation loss and perplexity on the held-out evaluation split.

In [ ]:
# Robust evaluation that works even if callback state is not initialized for on_evaluate
if "trainer" not in globals() or "eval_dataset" not in globals():
    raise RuntimeError("Run Cells 12-14 first to build trainer and datasets.")

if len(eval_dataset) == 0:
    raise RuntimeError("Evaluation dataset is empty.")

try:
    pred_output = trainer.predict(eval_dataset, metric_key_prefix="eval")
    eval_metrics = pred_output.metrics
except Exception as exc:
    print(f"Predict-based evaluation failed: {exc}")
    print("Run Cell 14 (Train) again, then rerun this cell.")
    raise

eval_loss = eval_metrics.get("eval_loss") or eval_metrics.get("test_loss")

if eval_loss is not None:
    perplexity = math.exp(eval_loss) if eval_loss < 20 else float("inf")
    print(f"Eval loss: {eval_loss:.4f}")
    print(f"Perplexity: {perplexity:.4f}" if perplexity != float("inf") else "Perplexity: inf")
else:
    print("Eval loss not found in metrics.")

print("All eval metrics:")
print(eval_metrics)

## Step 5: Save adapter

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"Saved adapter and tokenizer to {OUTPUT_DIR}")
print("Set PEFT_ADAPTER_PATH_BACKLOG in AI/.env to use this adapter:")
print("  PEFT_ADAPTER_PATH_BACKLOG=llms/fine_tune/qwen_model2_backlog_lora")

## Step 6: Quick inference test

In [ ]:
import re
from copy import deepcopy

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
model_infer = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))
if torch.cuda.is_available():
    model_infer = model_infer.to("cuda")

sample_part1_text = """=== Task Management Web Application ===
Summary:
Task management web app for small teams with collaborative workflows and real-time updates.

Roles:
- Project Manager
- Backend Developer
- Frontend Developer
- QA Engineer

Features:
- Kanban Board
- Real-time updates
- Task assignment
- Notifications

Goals:
- Build task API (Backend Developer)
- Build Kanban dashboard (Frontend Developer)
- Implement real-time updates (Backend Developer)
- Add quality validation (QA Engineer)

Timeline:
Week 1: Setup project architecture, Define API contracts
Week 2: Implement task CRUD, Create dashboard UI
Week 3: Add websocket updates, Implement task assignment flow
Week 4: Execute system testing, Prepare release candidate
"""

test_prompt = f"""You are given a structured project overview in plain text. Generate backlog text ONLY using this exact hierarchy:\nGoal X: <Goal title>\n  -User Story X.1: <As a ..., I want ...>\n    -Task X.1.1: <Task description>\n    -Task X.1.2: <Task description>\n\nRules:\n- Output backlog text only (no JSON, no markdown fences, no explanation).\n- Create one Goal block per goal in the input.\n- Each Goal has exactly 1 User Story.\n- Each Goal has exactly 2 Tasks.\n- Do not output Status, Summary, Roles, Features, Timeline, or Proposal / Input.\n\nInput:\n{sample_part1_text}\n\nBacklog:\n"""

inputs = tokenizer(test_prompt, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}

gen_cfg = deepcopy(model_infer.generation_config)
gen_cfg.do_sample = False
gen_cfg.temperature = None
gen_cfg.top_p = None
gen_cfg.top_k = None

outputs = model_infer.generate(
    **inputs,
    generation_config=gen_cfg,
    max_new_tokens=768,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    repetition_penalty=1.05,
    no_repeat_ngram_size=3,
)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def _parse_goal_hierarchy(text: str) -> list[dict]:
    goals: list[dict] = []
    current: dict | None = None
    for raw in text.splitlines():
        s = raw.strip()
        goal_match = re.match(r"^Goal\s*(\d+)\s*:\s*(.+)$", s, re.IGNORECASE)
        if goal_match:
            if current is not None:
                goals.append(current)
            current = {
                "idx": int(goal_match.group(1)),
                "title": goal_match.group(2).strip(),
                "stories": [],
            }
            continue

        if current is None:
            continue

        story_match = re.match(r"^-?\s*User Story\s*[\d.]*\s*:\s*(.+)$", s, re.IGNORECASE)
        if story_match:
            current["stories"].append({"text": story_match.group(1).strip(), "tasks": []})
            continue

        task_match = re.match(r"^-?\s*Task\s*[\d.]*\s*:\s*(.+)$", s, re.IGNORECASE)
        if task_match and current["stories"]:
            current["stories"][-1]["tasks"].append(task_match.group(1).strip())

    if current is not None:
        goals.append(current)
    return goals

print("===== Quick Test Output (Model 2) =====")
print(f"Raw response length: {len(response)} chars")
print("\nFull backlog text:\n")
print(response)

parsed_goals = _parse_goal_hierarchy(response)
print("\n----- Contract Check -----")
print(f"Goals parsed: {len(parsed_goals)}")
for goal in parsed_goals:
    story_count = len(goal["stories"])
    task_count = sum(len(story["tasks"]) for story in goal["stories"])
    print(f"Goal {goal['idx']}: {goal['title']} | stories: {story_count}, tasks: {task_count}")
    if story_count != 1:
        print("  -> FAIL: each goal must have exactly 1 user story")
    if task_count != 2:
        print("  -> FAIL: each goal must have exactly 2 tasks")

for forbidden in ["Status:", "Summary:", "Roles:", "Features:", "Timeline:", "Proposal / Input:"]:
    if forbidden.lower() in response.lower():
        print(f"Forbidden section found: {forbidden}")